In [ ]:
"""
PERSONAL MEMORY AI SYSTEM

Description
-----------
Personal Memory AI is an intelligent memory management and analysis system
designed to transform large collections of personal notes into a structured,
searchable, and evolving knowledge base.

The system imports text notes, processes their contents using natural language
processing techniques, extracts meaningful information, and builds long-term
memories from the collected data.

Core Capabilities
-----------------

1. Note Management
   - Import thousands of text notes
   - Store notes in a structured database
   - Maintain timestamps and metadata
   - Generate note summaries

2. Memory Extraction
   - Extract goals
   - Extract projects
   - Extract beliefs and opinions
   - Extract observations
   - Create long-term memory records

3. Knowledge Organization
   - Topic extraction
   - Entity recognition
   - Relationship extraction
   - Knowledge graph construction
   - Concept network generation

4. Semantic Understanding
   - Generate vector embeddings
   - Build vector search indexes
   - Perform semantic similarity search
   - Retrieve relevant memories

5. Timeline Construction
   - Detect dates and events
   - Build chronological memory timelines
   - Track personal development over time

6. Pattern Discovery
   - Identify recurring interests
   - Detect common goals
   - Analyze project evolution
   - Discover behavioral patterns

7. Contradiction Analysis
   - Compare old and new beliefs
   - Detect conflicting information
   - Track changes in viewpoints

8. Personality Inference
   - Analyze long-term behavior
   - Infer learning interests
   - Identify project tendencies
   - Build a dynamic personality profile

9. Trend Analysis
   - Detect emerging topics
   - Detect declining interests
   - Analyze topic frequency changes
   - Monitor knowledge growth

10. Prediction System
    - Predict future interests
    - Predict likely learning directions
    - Predict project expansion areas

11. Question Answering
    - Answer questions about stored memories
    - Retrieve relevant notes
    - Search semantically related information
    - Generate memory-based responses

System Components
-----------------

Database Layer
    Stores notes, memories, entities, topics,
    relationships, embeddings, observations,
    and predictions.

Memory Layer
    Maintains knowledge graphs, timelines,
    memory caches, and concept networks.

NLP Layer
    Performs text cleaning, summarization,
    entity extraction, topic extraction,
    and relationship extraction.

Embedding Layer
    Converts notes into vector representations
    for semantic understanding and search.

Analysis Layer
    Performs pattern analysis, contradiction
    detection, trend analysis, personality
    inference, and prediction generation.

Application Flow
----------------

1. Import Notes
2. Clean Text
3. Store Notes
4. Generate Summaries
5. Score Importance
6. Extract Topics
7. Extract Goals
8. Extract Projects
9. Extract Beliefs
10. Extract Entities
11. Extract Relationships
12. Create Memories
13. Build Knowledge Graph
14. Build Timeline
15. Generate Embeddings
16. Store Embeddings
17. Analyze Patterns
18. Infer Personality
19. Analyze Trends
20. Predict Future Interests
21. Enable Interactive Question Answering

Goal
----

The objective of this system is to create a persistent,
self-improving personal memory repository capable of
understanding, organizing, analyzing, and retrieving
knowledge from large collections of personal notes.

Author:
    Vrushabh.S

Version:
    1.0.0

Status:
    Improvement
"""
# IMPORTS


import sqlite3
import re
import datetime
from pathlib import Path

# NLP
import nltk
import spacy

# Data Processing
import pandas as pd
import numpy as np

# Embeddings
from sentence_transformers import SentenceTransformer

# Similarity Search
import faiss

# Graph Processing
import networkx as nx

# Optional ML Utilities
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Tokenizer Download
nltk.download("punkt")
nltk.download("punkt_tab")

# Load spaCy model

try:
    nlp = spacy.load(
        "en_core_web_sm"
    )
except OSError:
    print(
        "Run: python -m spacy download en_core_web_sm"
    )
    raise


# DATABASE INITIALIZATION


DATABASE_PATH = "personal_memory.db"


def initialize_database():
    """
    Create all database tables if they do not exist.
    """

    conn = sqlite3.connect(DATABASE_PATH)
    cursor = conn.cursor()


    # NOTES
 

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS notes (
        note_id INTEGER PRIMARY KEY AUTOINCREMENT,
        content TEXT,
        summary TEXT,
        importance_score REAL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    # MEMORIES


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS memories (
        memory_id INTEGER PRIMARY KEY AUTOINCREMENT,
        memory_text TEXT,
        memory_type TEXT,
        confidence REAL,
        source_note_id INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        memory_score REAL,
        frequency INTEGER DEFAULT 1,
        last_seen TIMESTAMP
    )
    """)

    # TOPICS
    
    cursor.execute("""
CREATE TABLE IF NOT EXISTS topics (
    topic_id INTEGER PRIMARY KEY AUTOINCREMENT,
    topic_name TEXT,
    frequency INTEGER,
    source_note_id INTEGER,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
""")
    # ENTITIES
  

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS entities (
        entity_id INTEGER PRIMARY KEY AUTOINCREMENT,
        entity_name TEXT,
        entity_type TEXT,
        source_note_id INTEGER,
        frequency INTEGER DEFAULT 0,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    # RELATIONSHIPS
    
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS relationships (
        relationship_id INTEGER PRIMARY KEY AUTOINCREMENT,
        source_node TEXT,
        relation TEXT,
        target_node TEXT,
        confidence REAL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
    )
    """)

    # EMBEDDINGS
 

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS embeddings (
        embedding_id INTEGER PRIMARY KEY AUTOINCREMENT,
        note_id INTEGER,
        vector BLOB
    )
    """)

    # OBSERVATIONS


    cursor.execute("""
    CREATE TABLE IF NOT EXISTS observations (
        observation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        observation_text TEXT
    )
    """)

    # PREDICTIONS
  

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS predictions (
        prediction_id INTEGER PRIMARY KEY AUTOINCREMENT,
        prediction_text TEXT
    )
    """)

    conn.commit()
    conn.close()

    print("Database initialized successfully.")

# MEMORY INITIALIZATION


class MemorySystem:
    """
    Core in-memory structures used by the AI system.
    """

    def __init__(self):

        # Knowledge Graph
        self.knowledge_graph = nx.DiGraph()

        # Timeline Events
        self.timeline = []

        # Vector Search Index
        self.vector_index = None

        # Extracted Information
        self.beliefs = []
        self.goals = []
        self.projects = []

        # Knowledge Collections
        self.topics = []
        self.entities = []

        # Fast Memory Cache
        self.memory_cache = {}

        # Statistics
        self.total_notes = 0
        self.total_memories = 0

    def initialize_vector_index(self, embedding_dimension=384):
        """
        Create FAISS vector index.
        """

        self.vector_index = faiss.IndexFlatL2(
            embedding_dimension
        )

        print(
            f"Vector Index Initialized "
            f"(Dimension={embedding_dimension})"
        )

    def display_status(self):

        print("\nMemory System Initialized")

        print(f"Notes: {self.total_notes}")
        print(f"Goals: {len(self.goals)}")
        print(f"Projects: {len(self.projects)}")
        print(f"Beliefs: {len(self.beliefs)}")
        print(f"Topics: {len(self.topics)}")
        print(f"Entities: {len(self.entities)}")


# Create Global Memory System

memory_system = MemorySystem()

memory_system.initialize_vector_index()


# NOTES IMPORT SYSTEM


NOTES_FOLDER = "notes"


def import_notes(notes_folder=NOTES_FOLDER):
    """
    Load all txt notes from folder.
    """

    notes_collection = []

    folder_path = Path(notes_folder)

    if not folder_path.exists():

        print(f"Folder not found: {notes_folder}")

        return notes_collection

    txt_files = list(
        folder_path.rglob("*.txt")
    )

    print(
        f"Found {len(txt_files)} text files."
    )

    for file_path in txt_files:

        try:

            with open(
                file_path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as file:

                content = file.read()

            content = content.encode(
                "utf-8",
                errors="ignore"
            ).decode("utf-8")

            content = re.sub(
                r"[\x00-\x1F\x7F]",
                " ",
                content
            )

            note_data = {
                "file_name": file_path.name,
                "file_path": str(file_path),
                "content": content,
                "length": len(content),
                "import_time":
                    datetime.datetime.now()
                    .isoformat()
            }

            notes_collection.append(
                note_data
            )

            print(
                f"Imported: {file_path.name}"
            )

        except Exception as error:

            print(
                f"Failed: {file_path.name}"
            )

            print(error)

    print(
        f"\nSuccessfully Imported "
        f"{len(notes_collection)} Notes"
    )

    return notes_collection

# NOTES STORAGE SYSTEM

def store_note(
    content,
    summary="",
    importance_score=0
):
    """
    Store a note in the database.
    """

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    cursor.execute(
        """
        INSERT INTO notes
        (
            content,
            summary,
            importance_score
        )
        VALUES (?, ?, ?)
        """,
        (
            content,
            summary,
            importance_score
        )
    )

    note_id = cursor.lastrowid

    conn.commit()

    conn.close()

    memory_system.total_notes += 1

    return note_id

# TEXT CLEANING SYSTEM


def clean_text(text):
    """
    Clean raw note text.
    """

    if not text:

        return ""

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(
        r"http\S+|www\S+",
        " ",
        text
    )

    # Remove email addresses
    text = re.sub(
        r"\S+@\S+",
        " ",
        text
    )

    # Remove special characters
    text = re.sub(
        r"[^a-z0-9\s]",
        " ",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    # Remove leading/trailing spaces
    text = text.strip()

    return text

# NOTES SUMMARIZATION SYSTEM

from nltk.tokenize import sent_tokenize


embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


def generate_summary(
    text,
    max_sentences=3
):
    """
    Generate summary using semantic
    sentence ranking.
    """

    # Empty text protection
    if not text or not text.strip():
        return ""

    try:

        sentences = sent_tokenize(
            text
        )

        # No sentences found
        if len(sentences) == 0:
            return ""

        # Small note
        if len(sentences) <= max_sentences:
            return text

        sentence_embeddings = (
            embedding_model.encode(
                sentences,
                convert_to_numpy=True
            )
        )

        document_embedding = (
            embedding_model.encode(
                text,
                convert_to_numpy=True
            )
        )

        similarities = (
            cosine_similarity(
                [document_embedding],
                sentence_embeddings
            )[0]
        )

        ranked_indices = (
            np.argsort(
                similarities
            )[::-1]
        )

        selected_indices = sorted(
            ranked_indices[
                :max_sentences
            ]
        )

        summary = " ".join(
            sentences[index]
            for index in selected_indices
        )

        return summary

    except Exception as error:

        print(
            f"Summary Error: {error}"
        )

        return text[:500]
    
# IMPORTANCE SCORING SYSTEM


def calculate_importance(
    text
):
    
    if not text.strip():
        return 0.0

    """
    Calculate importance using
    multiple semantic signals.
    """

    score = 0.0

    word_count = len(
        text.split()
    )

    # Information Density

    score += min(
        word_count / 10,
        25
    )

    # Uniqueness

    unique_words = len(
        set(text.split())
    )

    diversity = (
        unique_words /
        max(word_count, 1)
    )

    score += diversity * 20

    # Entity Density

    doc = nlp(
    text[:100000]
)

    entity_count = len(
        doc.ents
    )

    score += min(
        entity_count * 2,
        20
    )

    # Semantic Complexity

    sentence_count = len(
        list(doc.sents)
    )

    score += min(
        sentence_count,
        15
    )

    # Novelty

    score += min(
    diversity * 10,
    10
)

    return round(
        min(score, 100),
        2
    )



# TOPIC EXTRACTION SYSTEM

from sklearn.feature_extraction.text import TfidfVectorizer


def extract_topics(
    text,
    top_k=10
):
    """
    Extract important topics
    from text using TF-IDF.
    """

    # Empty text protection
    if not text or not text.strip():
        return []

    try:

        vectorizer = TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 3),
            max_features=500
        )

        tfidf_matrix = (
            vectorizer.fit_transform(
                [text]
            )
        )

        feature_names = (
            vectorizer.get_feature_names_out()
        )

        scores = (
            tfidf_matrix
            .toarray()[0]
        )

        ranked_indices = (
            np.argsort(scores)[::-1]
        )

        topics = []

        for index in ranked_indices[:top_k]:

            if scores[index] > 0:

                topics.append(
                    feature_names[index]
                )

        return topics

    except ValueError:
        # Happens when text contains
        # only stopwords or no valid words
        return []

    except Exception as error:

        print(
            f"Topic Extraction Error: {error}"
        )

        return []

# GOAL EXTRACTION SYSTEM

def extract_goals(text):
    """
    Extract goal-like statements.

    Placeholder until a local LLM
    is integrated.
    """

    # Empty text protection
    if not text or not text.strip():
        return []

    sentences = nltk.sent_tokenize(
        text
    )

    # No sentences found
    if len(sentences) == 0:
        return []

    # Single sentence note
    if len(sentences) == 1:

        if len(sentences[0].split()) > 5:
            return [
                sentences[0].strip()
            ]

        return []

    try:

        embeddings = embedding_model.encode(
            sentences,
            convert_to_numpy=True
        )

        document_embedding = (
            embedding_model.encode(
                text,
                convert_to_numpy=True
            )
        )

        similarities = (
            cosine_similarity(
                [document_embedding],
                embeddings
            )[0]
        )

        ranked_indices = np.argsort(
            similarities
        )[::-1]

        goals = []

        for index in ranked_indices[:5]:

            sentence = (
                sentences[index]
                .strip()
            )

            if len(sentence.split()) > 5:

                goals.append(
                    sentence
                )

        return goals

    except Exception as error:

        print(
            f"Goal Extraction Error: {error}"
        )

        return []


# PROJECT EXTRACTION SYSTEM


def extract_projects(
    text,
    max_projects=5
):

    if not text or not text.strip():
        return []

    sentences = nltk.sent_tokenize(
        text
    )

    if len(sentences) == 0:
        return []
    """
    Extract project-like statements
    from a note.
    """

    if not text or not text.strip():
        return []

    try:

        sentences = nltk.sent_tokenize(
            text
        )

        if len(sentences) == 0:
            return []

        if len(sentences) == 1:

            sentence = (
                sentences[0]
                .strip()
            )

            if len(sentence.split()) >= 6:
                return [sentence]

            return []

        sentence_embeddings = (
            embedding_model.encode(
                sentences,
                convert_to_numpy=True
            )
        )

        document_embedding = (
            embedding_model.encode(
                text,
                convert_to_numpy=True
            )
        )

        similarities = (
            cosine_similarity(
                [document_embedding],
                sentence_embeddings
            )[0]
        )

        ranked_indices = np.argsort(
            similarities
        )[::-1]

        projects = []
        seen = set()

        for index in ranked_indices:

            sentence = (
                sentences[index]
                .strip()
            )

            if len(sentence.split()) < 6:
                continue

            if sentence in seen:
                continue

            projects.append(
                sentence
            )

            seen.add(
                sentence
            )

            if len(projects) >= max_projects:
                break

        return projects

    except Exception as error:

        print(
            f"Project Extraction Error: {error}"
        )

        return []

# ENTITY EXTRACTION SYSTEM


def extract_entities(text):
    """
    Extract named entities from text.
    """

    doc = nlp(
    text[:100000]
)

    entities = []

    seen = set()

    for entity in doc.ents:

        entity_data = {
            "name": entity.text.strip(),
            "type": entity.label_
        }

        key = (
            entity_data["name"],
            entity_data["type"]
        )

        if key not in seen:

            entities.append(
                entity_data
            )

            seen.add(key)

    return entities

# RELATIONSHIP EXTRACTION SYSTEM


def extract_relationships(text):
    """
    Build simple relationships from
    entity co-occurrence.
    """

    doc = nlp(
    text[:100000]
)

    relationships = []

    for sentence in doc.sents:

        sentence_entities = []

        for entity in sentence.ents:

            sentence_entities.append(
                entity.text
            )

        if len(sentence_entities) < 2:

            continue

        for i in range(
            len(sentence_entities)
        ):

            for j in range(
                i + 1,
                len(sentence_entities)
            ):

                relationships.append(
                    {
                        "source":
                            sentence_entities[i],

                        "relation":
                            "RELATED_TO",

                        "target":
                            sentence_entities[j]
                    }
                )

    return relationships

# extract_belief_system

def extract_beliefs(text):

    doc = nlp(text)

    beliefs = []

    for sentence in doc.sents:

        sentence_text = (
            sentence.text.strip()
        )

        if any(
            phrase in sentence_text.lower()
            for phrase in
            [
                "i think",
                "i believe",
                "i feel",
                "in my opinion",
                "my view"
            ]
        ):
            beliefs.append(
                sentence_text
            )

    return beliefs

# MEMORY CREATION SYSTEM


def create_memories(
    note_id,
    topics,
    goals,
    projects,
    beliefs
):
    """
    Create memory records.
    """

    memories = []

    for topic in topics:

        memories.append(
            {
                "memory_text": topic,
                "memory_type": "TOPIC",
                "confidence": 0.80,
                "source_note_id": note_id
            }
        )

    for goal in goals:

        memories.append(
            {
                "memory_text": goal,
                "memory_type": "GOAL",
                "confidence": 0.90,
                "source_note_id": note_id
            }
        )

    for project in projects:

        memories.append(
            {
                "memory_text": project,
                "memory_type": "PROJECT",
                "confidence": 0.90,
                "source_note_id": note_id
            }
        )

    for belief in beliefs:

        memories.append(
            {
                "memory_text": belief,
                "memory_type": "BELIEF",
                "confidence": 0.75,
                "source_note_id": note_id
            }
        )

    return memories

# SAVE MEMORIES SYSTEM

def save_memories(memories):

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    for memory in memories:

        cursor.execute(
            """
            INSERT INTO memories
            (
                memory_text,
                memory_type,
                confidence,
                source_note_id
            )
            VALUES (?, ?, ?, ?)
            """,
            (
                memory["memory_text"],
                memory["memory_type"],
                memory["confidence"],
                memory["source_note_id"]
            )
        )

    conn.commit()

    memory_system.total_memories += len(
        memories
    )

    conn.close()

# KNOWLEDGE GRAPH BUILDER


def build_graph(
    entities,
    relationships
):
    """
    Build knowledge graph.
    """

    graph = (
        memory_system
        .knowledge_graph
    )

    for entity in entities:

        graph.add_node(
            entity["name"],
            entity_type=
            entity["type"]
        )

    for relationship in relationships:

        graph.add_edge(
            relationship["source"],
            relationship["target"],
            relation=
            relationship["relation"]
        )

    return graph

def save_relationships(
    relationships
):

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    for relationship in relationships:

        cursor.execute(
    """
    INSERT INTO relationships
    (
        source_node,
        relation,
        target_node,
        confidence,
        created_at
    )
    VALUES (?, ?, ?, ?, ?)
    """,
    (
     (
        relationship["source"],
        relationship["relation"],
        relationship["target"],
        0.8,
        datetime.datetime.now()
     )
    )
)
        

    conn.commit()

    conn.close()

# TIMELINE BUILDER


def build_timeline(
    note_id,
    text
):
    """
    Extract timeline events.
    """

    doc = nlp(
    text[:100000]
)

    events = []

    for entity in doc.ents:

        if entity.label_ == "DATE":

            events.append(
                {
                    "note_id": note_id,
                    "date": entity.text,
                    "event": text[:200]
                }
            )

    for event in events:

     if event not in memory_system.timeline:

        memory_system.timeline.append(
            event
        )

    return events

# EMBEDDING GENERATION

def generate_embedding(
    text
):
    """
    Convert text into vector.

    """

    if not text or not text.strip():
     return np.zeros(
        embedding_model.get_embedding_dimension(),
        dtype=np.float32
    )

    vector = embedding_model.encode(
        text,
        convert_to_numpy=True
    )

    return vector

# VECTOR STORAGE


note_id_to_index = {}

index_to_note_id = {}


def store_embedding(
    note_id,
    vector
):

    vector32 = np.array(
        [vector]
    ).astype(
        "float32"
    )

    current_index = (
        memory_system
        .vector_index
        .ntotal
    )

    memory_system.vector_index.add(
        vector32
    )

    note_id_to_index[
        note_id
    ] = current_index

    index_to_note_id[
        current_index
    ] = note_id

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    cursor.execute(
        """
        INSERT INTO embeddings
        (
            note_id,
            vector
        )
        VALUES (?, ?)
        """,
        (
            note_id,
            vector.astype(
    np.float32
).tobytes()
        )
    )

    conn.commit()
    conn.close()

# SAVE ENTITIES SYSTEM

def save_entities(
    note_id,
    entities
):

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    for entity in entities:

        cursor.execute(
            """
            INSERT INTO entities
            (
                entity_name,
                entity_type,
                source_note_id,
                frequency
            )
            VALUES (?, ?, ?, ?)
            """,
            (
                entity["name"],
                entity["type"],
                note_id,
                1
            )
        )

    conn.commit()
    conn.close()

# SEMANTIC SEARCH ENGINE

def search_memory(
    query,
    top_k=5
):
    """
    Search memories using
    semantic embeddings.
    """

    # Empty query protection
    if not query or not query.strip():
        return []

    # No vectors stored yet
    if (
        memory_system.vector_index is None
        or
        memory_system.vector_index.ntotal == 0
    ):
        return []

    try:

        query_vector = (
            generate_embedding(
                query
            )
        )

        query_vector = np.array(
            [query_vector]
        ).astype(
            "float32"
        )

        actual_k = min(
            top_k,
            memory_system.vector_index.ntotal
        )

        distances, indices = (
            memory_system
            .vector_index
            .search(
                query_vector,
                actual_k
            )
        )

        results = []

        conn = sqlite3.connect(
            DATABASE_PATH
        )

        cursor = conn.cursor()

        for index in indices[0]:

            if index == -1:
                continue

            note_id = (
                index_to_note_id
                .get(index)
            )

            if note_id is None:
                continue

            cursor.execute(
                """
                SELECT
                    note_id,
                    content,
                    summary,
                    importance_score
                FROM notes
                WHERE note_id = ?
                """,
                (note_id,)
            )

            row = cursor.fetchone()

            if row:

                results.append(
                    {
                        "note_id": row[0],
                        "content": row[1],
                        "summary": row[2],
                        "importance": row[3]
                    }
                )

        conn.close()

        return results

    except Exception as error:

        print(
            f"Search Error: {error}"
        )

        return []

# CONTRADICTION DETECTOR


def detect_contradictions(
    beliefs
):
    """
    Detect potentially conflicting beliefs.
    """

    contradictions = []

    if len(beliefs) < 2:

        return contradictions

    belief_embeddings = (
        embedding_model.encode(
            beliefs,
            convert_to_numpy=True
        )
    )

    for i in range(
        len(beliefs)
    ):

        for j in range(
            i + 1,
            len(beliefs)
        ):

            similarity = (
                cosine_similarity(
                    [belief_embeddings[i]],
                    [belief_embeddings[j]]
                )[0][0]
            )

            if similarity < 0.20:

                contradictions.append(
                    {
                        "belief_1":
                            beliefs[i],

                        "belief_2":
                            beliefs[j],

                        "similarity":
                            float(
                                similarity
                            )
                    }
                )

    return contradictions

# PATTERN ANALYSIS ENGINE


def analyze_patterns():
    """
    Discover recurring themes.
    """

    observations = []

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT topic_name,
               frequency
        FROM topics
        ORDER BY frequency DESC
        """
    )

    topic_rows = (
        cursor.fetchall()
    )

    for row in topic_rows:

        topic = row[0]
        frequency = row[1]

        observations.append(
            f"Topic '{topic}' "
            f"appeared "
            f"{frequency} times."
        )

    cursor.close()

    conn.close()

    return observations

# PERSONALITY INFERENCE ENGINE


def infer_personality():
    """
    Build personality profile
    from accumulated memories.
    """

    profile = []

    goal_count = len(
        memory_system.goals
    )

    project_count = len(
        memory_system.projects
    )

    topic_count = len(
        memory_system.topics
    )

    memory_count = (
        memory_system.total_memories
    )

    if project_count >= 5:

        profile.append(
            "LONG_TERM_BUILDER"
        )

    if goal_count >= 5:

        profile.append(
            "FUTURE_PLANNER"
        )

    if topic_count >= 20:

        profile.append(
            "ACTIVE_LEARNER"
        )

    if memory_count >= 100:

        profile.append(
            "KNOWLEDGE_COLLECTOR"
        )

    if not profile:

        profile.append(
            "INSUFFICIENT_DATA"
        )

    return profile

# CONCEPT NETWORK BUILDER


def build_concept_network():
    """
    Connect related topics.
    """

    concept_network = (
        nx.Graph()
    )

    topics = (
        memory_system.topics
    )

    for topic in topics:

        concept_network.add_node(
            topic
        )

    for i in range(
        len(topics)
    ):

        for j in range(
            i + 1,
            len(topics)
        ):

            topic_a = topics[i]
            topic_b = topics[j]

            emb_a = (
                generate_embedding(
                    topic_a
                )
            )

            emb_b = (
                generate_embedding(
                    topic_b
                )
            )

            similarity = (
                cosine_similarity(
                    [emb_a],
                    [emb_b]
                )[0][0]
            )

            if similarity > 0.70:

                concept_network.add_edge(
                    topic_a,
                    topic_b,
                    weight=similarity
                )

    return concept_network

# MEMORY SCORING SYSTEM


def score_memory(
    importance,
    confidence,
    frequency=1,
    recency_days=0
):
    """
    Calculate final memory score.
    """

    importance_weight = (
        importance * 0.40
    )

    confidence_weight = (
        confidence * 100 * 0.30
    )

    frequency_weight = (
        min(
            frequency,
            20
        ) * 1.5
    )

    recency_weight = max(
        0,
        20 - recency_days
    )

    final_score = (
        importance_weight
        + confidence_weight
        + frequency_weight
        + recency_weight
    )

    return round(
        final_score,
        2
    )

# TREND ANALYSIS ENGINE


def analyze_trends():
    """
    Analyze topic trends.
    """

    trends = []

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    cursor.execute(
        """
        SELECT
            topic_name,
            frequency
        FROM topics
        ORDER BY frequency DESC
        """
    )

    rows = cursor.fetchall()

    if not rows:

        return trends

    highest_frequency = (
        rows[0][1]
    )

    if highest_frequency == 0:

        conn.close()
        
        return []
    
    for topic, frequency in rows:

        percentage = (
            frequency /
            highest_frequency
        ) * 100

        trends.append(
            {
                "topic": topic,
                "frequency": frequency,
                "trend_strength":
                    round(
                        percentage,
                        2
                    )
            }
        )

    conn.close()

    return trends

# PREDICTION ENGINE


def predict_next_interest():
    """
    Predict future interests.
    """

    trends = analyze_trends()

    if len(trends) == 0:

        return (
            "No prediction available."
        )

    top_trends = trends[:5]

    predicted_topics = []

    for trend in top_trends:

        predicted_topics.append(
            trend["topic"]
        )

    prediction = {
        "predicted_interests":
            predicted_topics,
        "confidence":
            round(
                len(
                    predicted_topics
                ) / 5,
                2
            )
    }

    return prediction

# topic_save

def save_topics(
    note_id,
    topics
):

    conn = sqlite3.connect(
        DATABASE_PATH
    )

    cursor = conn.cursor()

    for topic in topics:

        cursor.execute(
            """
            INSERT INTO topics
            (
                topic_name,
                frequency,
                source_note_id
            )
            VALUES (?, ?, ?)
            """,
            (
                topic,
                1,
                note_id
            )
        )

    conn.commit()
    conn.close()

# ASKING SYSTEM

def ask(question):

    results = search_memory(
        question,
        top_k=5
    )

    if not results:
        return "No relevant memories found."

    answer = []

    for result in results:

        if result["summary"]:
            answer.append(
                result["summary"]
            )
        else:
            answer.append(
                result["content"][:500]
            )

    return "\n\n".join(answer)



# POST PROCESSING


def run_post_processing():

    print(
        "\nRunning Analysis..."
    )

    contradictions = (
        detect_contradictions(
            memory_system.beliefs
        )
    )

    observations = (
        analyze_patterns()
    )

    personality_profile = (
        infer_personality()
    )

    concept_network = (
        build_concept_network()
    )

    trends = (
        analyze_trends()
    )

    prediction = (
        predict_next_interest()
    )

    return {
        "contradictions":
            contradictions,

        "observations":
            observations,

        "personality":
            personality_profile,

        "concept_network":
            concept_network,

        "trends":
            trends,

        "prediction":
            prediction
    }

# OUTPUT RESULTS


def display_results(
    results
):

    print(
        "\nOBSERVATIONS"
    )

    for item in results[
        "observations"
    ]:

        print(
            f"- {item}"
        )

    print(
        "\nPERSONALITY PROFILE"
    )

    for trait in results[
        "personality"
    ]:

        print(
            f"- {trait}"
        )

    print(
        "\nTIMELINE EVENTS"
    )

    for event in (
        memory_system.timeline[:20]
    ):

        print(event)

    print(
        "\nKNOWLEDGE GRAPH"
    )

    print(
        "Nodes:",
        memory_system
        .knowledge_graph
        .number_of_nodes()
    )

    print(
        "Edges:",
        memory_system
        .knowledge_graph
        .number_of_edges()
    )

    print(
        "\nMEMORY STATISTICS"
    )

    print(
        f"Total Notes: "
        f"{memory_system.total_notes}"
    )

    print(
        f"Total Memories: "
        f"{memory_system.total_memories}"
    )

    print(
        "\nCONTRADICTIONS"
    )

    for contradiction in (
        results[
            "contradictions"
        ][:10]
    ):

        print(
            contradiction
        )

    print(
        "\nTREND ANALYSIS"
    )

    for trend in (
        results[
            "trends"
        ][:10]
    ):

        print(
            trend
        )

    print(
        "\nNEXT INTEREST PREDICTION"
    )

    print(
        results[
            "prediction"
        ]
    )

# INTERACTIVE MODE


def interactive_mode():

    print(
        "\nInteractive Mode Started"
    )

    print(
        "Type 'exit' to quit."
    )

    while True:

        question = input(
            "\nAsk: "
        )

        if (
            question.lower()
            == "exit"
        ):

            print(
                "\nGoodbye."
            )

            break

        answer = ask(
            question
        )

        print(
            "\nAnswer:\n"
        )

        print(answer)

# MEMORY PIPELINE

def run_memory_pipeline():

    


    notes = import_notes()

    for note in notes:

        cleaned_text = clean_text(
            note["content"]
        )

        summary = generate_summary(
    note["content"]
)

        importance = calculate_importance(
            cleaned_text
        )

        note_id = store_note(
            cleaned_text,
            summary,
            importance
        )

        topics = extract_topics(
            cleaned_text
        )

        goals = extract_goals(
            cleaned_text
        )

        projects = extract_projects(
            cleaned_text
        )

        memory_system.topics.extend(
    topics
        )

        memory_system.goals.extend(
    goals
)

        memory_system.projects.extend(
    projects
)

        

        beliefs = extract_beliefs(
    note["content"]
)
        
        memory_system.beliefs.extend(
    beliefs
)

        entities = extract_entities(
    note["content"]
)

    save_entities(
    note_id,
    entities
)

    for entity in entities:
     if entity["name"] not in memory_system.entities:
        memory_system.entities.append(
            entity["name"]
        )

    relationships = extract_relationships(
    note["content"]
)

    memories = create_memories(
            note_id,
            topics,
            goals,
            projects,
            beliefs
        )

    save_memories(memories)

    build_graph(
            entities,
            relationships
        )

    save_relationships(
            relationships
        )
        

    build_timeline(
    note_id,
    note["content"]
)

    vector = generate_embedding(
            cleaned_text
        )

    store_embedding(
            note_id,
            vector
        )

    save_topics(
    note_id,
    topics
)

    print("Pipeline Complete")
        

# APPLICATION START


if __name__ == "__main__":

    initialize_database()

    run_memory_pipeline()

    results = (
        run_post_processing()
    )

    display_results(
        results
    )

    interactive_mode()